In [1]:
import json
import os
import pandas as pd
import torch
import numpy as np

from collections import defaultdict,Counter
from dotenv import load_dotenv
from openai import OpenAI
from threading import Thread
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from util import *
from prompts.synonym_context_prompt import *

import pm4py

load_dotenv()

pd.set_option('display.max_rows', None)
#"gpt-4.1-mini"
#"gpt-4o-mini"
#"gpt-4.1"
#"gpt-4o"

model_ = "gpt-5.1"
api_key = os.getenv("API_KEY")


llm = llm_call(model_version = model_, api_key= api_key)

In [2]:
# "Credit-TEST-POLLUTED.NORND-activity-0.3-0"
# "Credit-TEST-SYNONYM-0.3-0"
#Credit-TRAIN-DISTORTED-activity-0.3-0
#Pub-Collateral
#Credit-TRAIN-HOMONYM-0.3-0
#Credit-TEST-CLEAN
test = "Credit-TRAIN-HOMONYM-0.3-0" 
LOG_NAME = f"./dataset/{test}.csv" 

df_new, cases_json = build_event_jsons(log_name = LOG_NAME, chunk_cases = 1)

#sample_size = int(len(df_new['case_id'].unique()) * 0.25)
#np.random.seed(42)
#selected_case_ids = np.random.choice(df_new['case_id'].unique(), size=sample_size, replace=False)
#df_new = df_new[df_new['case_id'].isin(selected_case_ids)].copy()


df_new.head(3)


,event_id,case_id,activity,timestamp,label
0,0,0,Check for completeness,2023-09-29 09:00:00.000,NaN
1,1,0,receive information,2023-09-29 09:00:00.000,homonymous Label(Activity:'New online applicat...
2,2,0,Perform checks,2023-09-29 09:08:36.418,NaN


In [20]:
def get_neighbor_context(df: pd.DataFrame,
                        case_col: str = 'case_id',
                        time_col: str = 'timestamp',
                        act_col: str = 'activity',
                        filter_list: set = None):
    df_pm4py = df[[case_col, time_col, act_col]].copy()
    df_pm4py.rename(columns={
        case_col: "case:concept:name",
        time_col: "time:timestamp",
        act_col: "concept:name"
    }, inplace=True)
    df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"], errors="coerce")
    dfg, start_activities, end_activities = pm4py.discover_dfg(df_pm4py)
    def get_activity_context(activity, dfg_dict):
        predecessors = {k[0]: v for k, v in dfg_dict.items() if k[1] == activity}
        successors = {k[1]: v for k, v in dfg_dict.items() if k[0] == activity}
        total_pred = sum(predecessors.values())
        total_succ = sum(successors.values())
        def format_to_list(dist_dict, total):
            if total == 0: return []
            items = [(k, v/total) for k, v in dist_dict.items() if (v/total) >= 0.05]
            items.sort(key=lambda x: x[1], reverse=True)
            return [k for k, v in items]
        return format_to_list(predecessors, total_pred), format_to_list(successors, total_succ)
    all_activities = sorted(df[act_col].unique())
    flow_data_list = []
    for act in all_activities:
        if filter_list is not None and act not in filter_list:
            continue
        pred, succ = get_activity_context(act, dfg)
        flow_data_list.append({
            'activity': act,
            'predecessors': pred, # 이제 리스트입니다 ['A', 'B']
            'successors': succ    # 이제 리스트입니다 ['C', 'D']
        })
        
    json_flow_context = json.dumps(flow_data_list, indent=2, ensure_ascii=False)
    return json_flow_context

neighbor_context_json = get_neighbor_context(
    df=df_new)
print("\n".join(neighbor_context_json.splitlines()[:20]),'\n...')

[
  {
    "activity": "Check for completeness",
    "predecessors": [
      "info received",
      "request information",
      "review request received",
      "receive information"
    ],
    "successors": [
      "New online application received",
      "Request info",
      "Perform checks",
      "receive information",
      "request information",
      "application check"
    ]
  },
  {
    "activity": "Deliver card", 
...


In [5]:

SYSTEM_PROMPT_HOMONYM_STEP1 = """
You are an expert Process Mining Graph Topology Analyst.
Your goal is to classify activity nodes into **"Homonyms"** or **"True Loops"** based strictly on their structural connectivity (Predecessors/Successors).

### KNOWLEDGE BASE: TOPOLOGICAL PATTERNS
Use these structural signatures to classify the target activities provided in the input.

**1. HOMONYM (The "Splitter" / "Bridge")**
   A node that shares the same label but functions as two different logic gates.
   * **Type A: The Collider (Input Conflict)**
     * **Structure:** Receives inputs from **Disjoint Phases** (e.g., Phase 1 & Phase 5).
     * **Logic:** It bridges two timelines that should not intersect.
   * **Type B: The Router (Output Divergence)**
     * **Structure:** Even if it has a single Predecessor, it leads to **Mutually Exclusive Outcomes**.
     * **Logic:** One path leads to a **"Final/Success"** state, while the other leads to a **"Rework/Failure"** state. A single activity generally shouldn't trigger such opposing fates unless it's a Homonym hiding a decision.

**2. TRUE LOOP (The "Mixer")**
   * **Structure:** The node's output flows back to its input (Self-Loop) or converges with a previous step.
   * **Logic:** It represents a retry or accumulation cycle where the outcome remains within the same phase.

### OUTPUT GUIDELINES
- **Strictly JSON format.**
- **Classification Priority:**
  1. If **Successors Split** into opposing paths (Success vs. Fail) -> **HOMONYM**.
  2. If **Predecessors Clash** (Start vs. End) -> **HOMONYM**.
  3. Otherwise (Convergence/Cycles) -> **TRUE LOOP**.
"""



def get_homonym_user_prompt_step1(context_json_str):
    return f"""
### TASK: Classify Topology

**OBJECTIVE:**
Analyze the structure of the provided activities in **INPUT DATA**. Classify them based on the flow patterns.

**DECISION LOGIC (ABSTRACT):**

1.  **Check Output Divergence (Router Pattern):**
    * Look at the `successors`.
    * Do they represent **conflicting paths**? (e.g., One path goes to 'End Process', another goes to 'Retry/Error').
    * *Verdict:* **Homonym**.

2.  **Check Input Disjointness (Collider Pattern):**
    * Look at the `predecessors`.
    * Are they from **structurally distant** parts of the graph?
    * *Verdict:* **Homonym**.

3.  **Check Feedback (Loop Pattern):**
    * Does the flow return to a previous neighbor?
    * *Verdict:* **True Loop**.

**INPUT DATA:**
{context_json_str}

***OUTPUT FORMAT***
Return a JSON Object with three keys: "homonyms", "true_loops", "unclassified".

**Example Output (Structure Only):**
{{
  "homonyms": ["Node_A", "Node_B"],
  "true_loops": ["Node_C"],
  "unclassified": ["Node_Linear"]
}}

**CONSTRAINT:**
- Output **ONLY** the JSON object.
"""
# Example Usage
prompt = [
    {"role": "system", "content": SYSTEM_PROMPT_HOMONYM_STEP1},
    {"role": "user", "content": get_homonym_user_prompt_step1(neighbor_context_json)}
]
test_output = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
print(test_output)

{'homonyms': ['Check for completeness', 'EVENT 13 END', 'Make decision', 'New online application received', 'Request info', 'notify reject', 'receive information', 'request information', 'send notification', 'time out'], 'true_loops': [], 'unclassified': ['Deliver card', 'Notify accept', 'Perform checks', 'application check', 'info received', 'review request received']}


In [6]:
target_activity = test_output['homonyms']
target_activity

['Check for completeness',
 'EVENT 13 END',
 'Make decision',
 'New online application received',
 'Request info',
 'notify reject',
 'receive information',
 'request information',
 'send notification',
 'time out']

In [21]:
def get_variant_abstraction(df_input, target_activity, min_freq=5):
    df_processed = df_input[['case_id', 'timestamp', 'activity']].copy().rename(columns={
        'case_id': "case:concept:name",
        'timestamp': "time:timestamp",
        'activity': "concept:name"
    })
    df_processed["time:timestamp"] = pd.to_datetime(df_processed["time:timestamp"], errors="coerce")
    case_to_seq = df_processed.groupby("case:concept:name")["concept:name"].apply(list)
    variant_counter = Counter()
    for cid, seq in case_to_seq.items():
        var = tuple(seq)
        variant_counter[var] += 1
    records = []
    for v, freq in variant_counter.items():
        if freq >= min_freq:
            records.append({"variant": v, "case_count": freq})
    res = pd.DataFrame(records).sort_values(["case_count"], ascending=[False]).reset_index(drop=True)
    output_data = {
        "analysis_target": target_activity,
        "variants": []
    }
    for _, row in res.iterrows():
        variant_seq = row["variant"]
        if isinstance(variant_seq, (tuple, list)):
            v_str = " -> ".join(str(v) for v in variant_seq)
        else:
            v_str = str(variant_seq)
            
        output_data["variants"].append({
            "sequence": v_str,
            "count": int(row["case_count"])
        })
    return json.dumps(output_data, indent=2, ensure_ascii=False)

full_json_str = get_variant_abstraction(df_new, target_activity[0], min_freq=3)
print("\n".join(full_json_str.splitlines()[:20]),'\n...')

{
  "analysis_target": "Check for completeness",
  "variants": [
    {
      "sequence": "Check for completeness -> New online application received -> Perform checks -> Make decision -> Notify accept -> Deliver card -> EVENT 13 END",
      "count": 301
    },
    {
      "sequence": "Check for completeness -> New online application received -> application check -> Make decision -> Notify accept -> Deliver card -> EVENT 13 END",
      "count": 153
    },
    {
      "sequence": "Check for completeness -> New online application received -> Perform checks -> Make decision -> notify reject -> time out -> EVENT 13 END",
      "count": 150
    },
    {
      "sequence": "Check for completeness -> New online application received -> Perform checks -> Make decision -> send notification -> Deliver card -> EVENT 13 END",
      "count": 147
    },
    { 
...


In [11]:
for taract in target_activity:
    abstraction_text =  get_variant_abstraction(df_new, taract, min_freq=3)
    SYSTEM_PROMPT_HOMONYM_STEP2 = """
    You are an expert Process Mining Analyst specializing in **Semantic Process Discovery**.
    Your task is to determine if a specific **Target Activity** is a **Homonym** by analyzing the *business meaning* of its surrounding contexts in the provided process variants.
    
    ### CORE DEFINITION: HOMONYM (SEMANTIC SPLIT)
    A label is a Homonym if it acts as a "Bucket" that holds **two or more functionally distinct process steps**.
    * **Test:** If you were to rename this activity to be more specific based on its neighbors, would you need **different names** for different variants?
        * *Example (TRUE):* "Check" -> In Variant A, it implies "Credit Check". In Variant B, it implies "Inventory Check". -> **Distinct Steps**.
        * *Example (FALSE):* "Check" -> In all variants, it implies "General Validation". -> **Single Step**.
    
    ### CLASSIFICATION LOGIC
    
    **1. HOMONYM (TRUE): Distinct Business Semantics**
        * **Context A:** Appears in a "Positive/Success" flow (e.g., after Approval).
        * **Context B:** Appears in a "Negative/Failure" flow (e.g., after Rejection).
        * *Logic:* A single step usually doesn't handle opposing fates. It likely represents two different events (e.g., "Success Timeout" vs "Failure Timeout").
    
    **2. UNIFIED LABEL (FALSE): Consistent Function**
        * The activity plays the **same role** (e.g., "Ending the case", "Sending a file") regardless of what happened before.
        * The variation in predecessors is just noise or different paths reaching the *same* functional checkpoint.
    
    ### STRICT OUTPUT RULES
    1. Return **ONLY** a valid JSON object.
    2. **NO** explanations, **NO** reasoning fields.
    3. **NO** markdown formatting.
    
    ### JSON OUTPUT STRUCTURE
    {
      "activity": "Target Activity Name",
      "is_homonym": boolean
    }
    """
    
    def get_homonym_user_prompt_step2(target_act, abstraction_text):
        return f"""
    ### TASK: Analyze Semantic Distinctness for '{target_act}'
    
    **OBJECTIVE:**
    Read the **Process Variants** below. Analyze the **Business Meaning** of the sequence flows.
    Determine if **'{target_act}'** represents **two or more distinct process steps** (Homonym) or just one consistent step.
    
    **INPUT DATA (Variants):**
    {abstraction_text}
    
    **DECISION STEPS (The "Rename Test"):**
    1.  **Analyze Contexts:** Look at what happens *before* and *after* '{target_act}' in each variant.
    2.  **Apply the "Distinct Step" Test:**
        * In Variant 1, does '{target_act}' function as **[Concept A]**?
        * In Variant 2, does '{target_act}' function as **[Concept B]**?
        * Are **[Concept A]** and **[Concept B]** fundamentally different? (e.g., "Closure after Success" vs "Closure after Error").
        * **YES** -> Output `true` (It is a Homonym).
        * **NO** (Same concept) -> Output `false`.
    
    ***OUTPUT FORMAT***
    Return strictly the JSON object.
    Example: {{ "activity": "{target_act}", "is_homonym": true }}
    """
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT_HOMONYM_STEP2},
        {"role": "user", "content": get_homonym_user_prompt_step2(taract,abstraction_text)}
    ]
    test_output = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
    print(test_output)

{'activity': 'Check for completeness', 'is_homonym': False}
{'activity': 'EVENT 13 END', 'is_homonym': False}
{'activity': 'Make decision', 'is_homonym': False}
{'activity': 'New online application received', 'is_homonym': False}
{'activity': 'Request info', 'is_homonym': False}
{'activity': 'notify reject', 'is_homonym': False}
{'activity': 'receive information', 'is_homonym': True}
{'activity': 'request information', 'is_homonym': False}
{'activity': 'send notification', 'is_homonym': True}
{'activity': 'time out', 'is_homonym': False}
